In [11]:
# Basic Package
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm_notebook

import warnings
warnings.filterwarnings("ignore")

In [12]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
data_dir = '/content/drive/MyDrive/hw02/pokemon' # Base Path, change your own path

IMG_SIZE = 64

transform = transforms.Compose([              #이미지 전처리 과정으로 임의로 수정 시
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  #차원이 달라질 수 있으니 수정하지 않는 것을 권장
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print("classes:", dataset.classes)
print("class_to_idx:", dataset.class_to_idx)

targets = torch.tensor([s[1] for s in dataset.samples])
num_classes = len(dataset.classes)

classes: ['bulbasaur', 'charizard', 'ditto', 'dragonite', 'gengar', 'mew', 'pikachu', 'quagsire', 'snorlax', 'squirtle']
class_to_idx: {'bulbasaur': 0, 'charizard': 1, 'ditto': 2, 'dragonite': 3, 'gengar': 4, 'mew': 5, 'pikachu': 6, 'quagsire': 7, 'snorlax': 8, 'squirtle': 9}


In [14]:
batch_size = 64

indices = np.arange(len(dataset))

k = 5 # k-fold 구현시 사용할 파라미터
seed = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


Device: cuda


In [15]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        running_correct += (preds == labels).sum().item()
        total += images.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = running_correct / total
    return epoch_loss, epoch_acc

In [21]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0

    all_preds = []
    all_labels = []

    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            total += images.size(0)

            _, preds = outputs.max(1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = running_loss / total

    import numpy as np
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")

    return avg_loss, acc, f1_macro

### class CNN(nn.Module)

- __ init __(self, ...): 모델 생성자(constructor), 레이어와 구조를 정의하는 부분
- forward(self, x): 순전파(forward pass), 입력이 모델을 통과하는 연산을 정의


### nn.Conv2d(in_channels, out_channels, kernel_size, ...)
- in_channels: 입력 채널 수 (예: RGB=3)
- out_channels: 필터 개수 = 출력 채널 수
- kernel_size: 필터 크기 (예: 3 → 3x3)
- padding: 가장자리를 얼마나 채울지 (예: 1 → 크기 유지)
- stride: 필터가 이동하는 간격

### nn.MaxPool2d(kernel_size, stride)
- kernel_size: 영역 크기 (예: 2 → 2x2 영역 중 최대값 선택)
- stride: 이동 간격 (보통 kernel_size와 같음)
- 역할: 가로·세로 크기를 절반으로 줄여 특징을 압축

### nn.Dropout(p)
- p: 뉴런을 0으로 만들 확률
- 역할: 과적합 방지

### nn.BatchNorm2d(num_features)
- num_features: 채널 수
- 역할: 각 배치의 평균/분산 기준으로 정규화하여 학습 안정화

### nn.Sigmoid()
- 역할: 입력을 0~1 사이 값으로 변환
- 공식: 1 / (1 + exp(-x))
- 특징: 이진 분류 등에서 확률 형태 출력에 사용


In [88]:
class CNN(nn.Module):
    def __init__(self, num_classes: int):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm2d(128)


        self.pool = nn.MaxPool2d(2, 2)

        self.dropout_conv = nn.Dropout(0.15)


        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.bn_fc = nn.BatchNorm1d(512)
        self.dropout_fc = nn.Dropout(0.3)

        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = torch.relu(self.bn4(self.conv4(x)))
        x = self.dropout_conv(x)

        x = x.view(x.size(0), -1)

        x = torch.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout_fc(x)
        x = self.fc2(x)

        return x


### KFold(n_splits)
- n_splits: 전체 데이터를 몇 개의 fold로 나눌지 (예: 5 → 5개 fold)
- 각 fold가 한 번씩 검증 데이터가 되고 나머지는 학습 데이터가 됨
- 모든 fold에 대해 학습 / 평가 후 성능을 평균함

In [92]:
KFold(n_splits=k, shuffle=True, random_state=seed)


KFold(n_splits=5, random_state=42, shuffle=True)

In [97]:
num_epochs = 15  # 20 이내로 수정 가능

criterion = nn.CrossEntropyLoss()
learning_rate = 1e-3   # 나중에 optimizer에서 사용

######여기를 채워넣으세요######
kfold = KFold(n_splits=k, shuffle=True, random_state=seed)
###############################

fold_acc_list = []
fold_f1_list  = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(indices), start=1):
    print(f"\n========== Fold {fold} / {k} ==========")

    train_subset = Subset(dataset, train_idx)
    test_subset   = Subset(dataset, test_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    test_loader   = DataLoader(test_subset,   batch_size=batch_size, shuffle=False)

    model = CNN(num_classes=num_classes).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

    for epoch in tqdm_notebook(range(num_epochs)):
        train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)

        print(f"[Fold {fold} | Epoch {epoch:02d}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
              f"|| Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1(macro): {test_f1:.4f}")

    fold_acc_list.append(test_acc)
    fold_f1_list.append(test_f1)


========== Fold 1 / 5 ==========


  0%|          | 0/15 [00:00<?, ?it/s]

[Fold 1 | Epoch 00] Train Loss: 1.3865 | Train Acc: 0.5347 || Test Loss: 2.4398 | Test Acc: 0.1111 | Test F1(macro): 0.0200
[Fold 1 | Epoch 01] Train Loss: 0.5110 | Train Acc: 0.8872 || Test Loss: 2.0606 | Test Acc: 0.2500 | Test F1(macro): 0.2180
[Fold 1 | Epoch 02] Train Loss: 0.2917 | Train Acc: 0.9358 || Test Loss: 1.0694 | Test Acc: 0.6458 | Test F1(macro): 0.6356
[Fold 1 | Epoch 03] Train Loss: 0.1542 | Train Acc: 0.9670 || Test Loss: 0.6388 | Test Acc: 0.8056 | Test F1(macro): 0.8195
[Fold 1 | Epoch 04] Train Loss: 0.0732 | Train Acc: 0.9948 || Test Loss: 0.2220 | Test Acc: 0.9514 | Test F1(macro): 0.9523
[Fold 1 | Epoch 05] Train Loss: 0.0397 | Train Acc: 0.9965 || Test Loss: 0.1822 | Test Acc: 0.9444 | Test F1(macro): 0.9462
[Fold 1 | Epoch 06] Train Loss: 0.0274 | Train Acc: 1.0000 || Test Loss: 0.1627 | Test Acc: 0.9653 | Test F1(macro): 0.9640
[Fold 1 | Epoch 07] Train Loss: 0.0155 | Train Acc: 1.0000 || Test Loss: 0.1733 | Test Acc: 0.9653 | Test F1(macro): 0.9671
[Fold 1 

  0%|          | 0/15 [00:00<?, ?it/s]

[Fold 2 | Epoch 00] Train Loss: 1.3598 | Train Acc: 0.5399 || Test Loss: 2.5729 | Test Acc: 0.1042 | Test F1(macro): 0.0284
[Fold 2 | Epoch 01] Train Loss: 0.5226 | Train Acc: 0.8698 || Test Loss: 2.7622 | Test Acc: 0.1250 | Test F1(macro): 0.0669
[Fold 2 | Epoch 02] Train Loss: 0.2391 | Train Acc: 0.9497 || Test Loss: 1.2940 | Test Acc: 0.5972 | Test F1(macro): 0.6392
[Fold 2 | Epoch 03] Train Loss: 0.1508 | Train Acc: 0.9757 || Test Loss: 0.5577 | Test Acc: 0.8264 | Test F1(macro): 0.8367
[Fold 2 | Epoch 04] Train Loss: 0.0748 | Train Acc: 0.9896 || Test Loss: 0.2362 | Test Acc: 0.9306 | Test F1(macro): 0.9325
[Fold 2 | Epoch 05] Train Loss: 0.0463 | Train Acc: 0.9965 || Test Loss: 0.1921 | Test Acc: 0.9306 | Test F1(macro): 0.9349
[Fold 2 | Epoch 06] Train Loss: 0.0250 | Train Acc: 0.9983 || Test Loss: 0.1483 | Test Acc: 0.9653 | Test F1(macro): 0.9656
[Fold 2 | Epoch 07] Train Loss: 0.0173 | Train Acc: 0.9983 || Test Loss: 0.1456 | Test Acc: 0.9444 | Test F1(macro): 0.9467
[Fold 2 

...

[Fold 3 | Epoch 00] Train Loss: 1.4089 | Train Acc: 0.5260 || Test Loss: 2.4809 | Test Acc: 0.0903 | Test F1(macro): 0.0166
[Fold 3 | Epoch 01] Train Loss: 0.6066 | Train Acc: 0.8385 || Test Loss: 2.1078 | Test Acc: 0.2292 | Test F1(macro): 0.2396
[Fold 3 | Epoch 02] Train Loss: 0.3180 | Train Acc: 0.9271 || Test Loss: 1.3975 | Test Acc: 0.5278 | Test F1(macro): 0.5500
[Fold 3 | Epoch 03] Train Loss: 0.1620 | Train Acc: 0.9688 || Test Loss: 0.5976 | Test Acc: 0.8403 | Test F1(macro): 0.8270
[Fold 3 | Epoch 04] Train Loss: 0.0919 | Train Acc: 0.9844 || Test Loss: 0.4655 | Test Acc: 0.8750 | Test F1(macro): 0.8813
[Fold 3 | Epoch 05] Train Loss: 0.0560 | Train Acc: 0.9913 || Test Loss: 0.3360 | Test Acc: 0.9306 | Test F1(macro): 0.9260
[Fold 3 | Epoch 06] Train Loss: 0.0422 | Train Acc: 0.9965 || Test Loss: 0.1757 | Test Acc: 0.9444 | Test F1(macro): 0.9410
[Fold 3 | Epoch 07] Train Loss: 0.0299 | Train Acc: 0.9983 || Test Loss: 0.2996 | Test Acc: 0.9236 | Test F1(macro): 0.9276
[Fold 3 

  0%|          | 0/15 [00:00<?, ?it/s]

[Fold 4 | Epoch 00] Train Loss: 1.3402 | Train Acc: 0.5521 || Test Loss: 2.2978 | Test Acc: 0.1042 | Test F1(macro): 0.0189
[Fold 4 | Epoch 01] Train Loss: 0.4439 | Train Acc: 0.9097 || Test Loss: 2.3547 | Test Acc: 0.1528 | Test F1(macro): 0.1011
[Fold 4 | Epoch 02] Train Loss: 0.1928 | Train Acc: 0.9653 || Test Loss: 1.5381 | Test Acc: 0.4167 | Test F1(macro): 0.4335
[Fold 4 | Epoch 03] Train Loss: 0.0892 | Train Acc: 0.9913 || Test Loss: 0.4929 | Test Acc: 0.8403 | Test F1(macro): 0.8533
[Fold 4 | Epoch 04] Train Loss: 0.0437 | Train Acc: 0.9983 || Test Loss: 0.2169 | Test Acc: 0.9375 | Test F1(macro): 0.9312
[Fold 4 | Epoch 05] Train Loss: 0.0316 | Train Acc: 0.9965 || Test Loss: 0.2018 | Test Acc: 0.9583 | Test F1(macro): 0.9480
[Fold 4 | Epoch 06] Train Loss: 0.0201 | Train Acc: 0.9983 || Test Loss: 0.2781 | Test Acc: 0.9167 | Test F1(macro): 0.9122
[Fold 4 | Epoch 07] Train Loss: 0.0188 | Train Acc: 0.9983 || Test Loss: 0.1988 | Test Acc: 0.9375 | Test F1(macro): 0.9308
[Fold 4 

  0%|          | 0/15 [00:00<?, ?it/s]

[Fold 5 | Epoch 00] Train Loss: 1.3729 | Train Acc: 0.5399 || Test Loss: 2.3315 | Test Acc: 0.1111 | Test F1(macro): 0.0390
[Fold 5 | Epoch 01] Train Loss: 0.5197 | Train Acc: 0.8698 || Test Loss: 2.7230 | Test Acc: 0.1250 | Test F1(macro): 0.0726
[Fold 5 | Epoch 02] Train Loss: 0.2368 | Train Acc: 0.9583 || Test Loss: 1.8506 | Test Acc: 0.3333 | Test F1(macro): 0.3469
[Fold 5 | Epoch 03] Train Loss: 0.1232 | Train Acc: 0.9809 || Test Loss: 0.6621 | Test Acc: 0.8194 | Test F1(macro): 0.8268
[Fold 5 | Epoch 04] Train Loss: 0.0687 | Train Acc: 0.9931 || Test Loss: 0.4172 | Test Acc: 0.8611 | Test F1(macro): 0.8648
[Fold 5 | Epoch 05] Train Loss: 0.0308 | Train Acc: 0.9983 || Test Loss: 0.2326 | Test Acc: 0.9444 | Test F1(macro): 0.9412
[Fold 5 | Epoch 06] Train Loss: 0.0199 | Train Acc: 1.0000 || Test Loss: 0.2798 | Test Acc: 0.9236 | Test F1(macro): 0.9192
[Fold 5 | Epoch 07] Train Loss: 0.0134 | Train Acc: 1.0000 || Test Loss: 0.1982 | Test Acc: 0.9514 | Test F1(macro): 0.9523
[Fold 5 

In [98]:
print("\n========== K-Fold 평균 결과 ==========")
print(f"Acc 평균: {np.mean(fold_acc_list):.4f}")
print(f"F1 평균: {np.mean(fold_f1_list):.4f}")


========== K-Fold 평균 결과 ==========
Acc 평균: 0.9653
F1 평균: 0.9651
